# `examples/examples_autoscaling` 命令行等价运行版

这个 Notebook 参考你之前能正常运行的样例 Notebook 写法，不在 Notebook 中重新拆开实现自动伸缩逻辑，而是直接以子进程方式执行原始 `.py` 文件：

```bash
python -u examples/examples_autoscaling/main.py
```

这样运行路径、日志行为和你在 PowerShell 中直接执行脚本基本一致。Notebook 只负责启动脚本、显示日志，并在运行结束后查看 `outputs/` 目录中的结果文件。

## 1. 定位项目根目录

下面的代码会从当前 Notebook 所在目录开始，向上寻找同时包含 `sim/` 和 `examples/` 的目录，并把它作为 faas-sim 项目根目录。

In [1]:
from pathlib import Path
import sys
import os
import subprocess

current_dir = Path.cwd().resolve()

candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent,
    current_dir.parent.parent.parent.parent,
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / "sim").exists() and (root / "examples").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "没有找到 faas-sim 项目根目录。请把 Notebook 放在项目根目录、"
        "examples/examples_autoscaling/ 或 examples/examples_autoscaling/notebook/ 下运行。"
    )

script_path = PROJECT_ROOT / "examples" / "examples_autoscaling" / "main.py"

print(f"当前 Notebook 工作目录：{current_dir}", flush=True)
print(f"faas-sim 项目根目录：{PROJECT_ROOT}", flush=True)
print(f"即将运行的样例脚本：{script_path}", flush=True)

if not script_path.exists():
    raise FileNotFoundError(f"找不到样例脚本：{script_path}")

当前 Notebook 工作目录：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\examples_autoscaling\notebook
faas-sim 项目根目录：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master
即将运行的样例脚本：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\examples_autoscaling\main.py


## 2. 使用当前 Jupyter 内核对应的 Python 运行样例

这里使用 `sys.executable`，保证 Notebook 使用哪个 Python 内核，就用哪个 Python 来执行样例脚本。

`-u` 参数表示 unbuffered，作用是让 Python 日志和 print 输出尽快刷新出来，尽量接近你在命令行中看到的效果。

In [2]:
print("当前 Jupyter 内核 Python：", sys.executable, flush=True)
print("开始以命令行等价方式运行 examples_autoscaling 样例。", flush=True)

当前 Jupyter 内核 Python： d:\miniconda3\python.exe
开始以命令行等价方式运行 examples_autoscaling 样例。


## 3. 执行 `examples/examples_autoscaling/main.py`

这一格会实时打印子进程输出。

如果 `.py` 脚本在 PowerShell 中能跑通，这一格通常也应该能跑通，并显示类似：

```text
INFO:__main__:creating autoscaling topology
INFO:sim.faassim:initializing simulation...
INFO:sim.faas.system:deploying function autoscale-python-pi with scale_min=1
INFO:__main__:triggering autoscaling workload
INFO:analysis:saved ... outputs/autoscaling_summary.csv
```

In [3]:
env = os.environ.copy()

# 确保子进程优先从项目根目录导入 sim、examples、ether、skippy、simpy 等本地包。
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    str(PROJECT_ROOT)
    if not existing_pythonpath
    else str(PROJECT_ROOT) + os.pathsep + existing_pythonpath
)

# 减少 Windows / Jupyter 下输出缓冲问题。
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-u",
    str(script_path),
]

print("执行命令：", " ".join(cmd), flush=True)
print("工作目录：", PROJECT_ROOT, flush=True)
print("开始输出子进程日志：", flush=True)

process = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

# 实时转发子进程输出。
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print(f"\n子进程退出码：{return_code}", flush=True)

if return_code != 0:
    raise RuntimeError(f"examples_autoscaling 样例运行失败，退出码：{return_code}")
else:
    print("examples_autoscaling 样例运行完成。", flush=True)

执行命令： d:\miniconda3\python.exe -u C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\examples_autoscaling\main.py
工作目录： C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master
开始输出子进程日志：
INFO:__main__:creating autoscaling topology
INFO:__main__:creating autoscaling benchmark
INFO:__main__:creating simulation
INFO:__main__:running simulation
INFO:sim.faassim:initializing simulation, benchmark: AutoscalingBenchmark, topology nodes: 117
INFO:system:creating autoscaling DefaultFaasSystem
INFO:sim.faassim:starting resource monitor
INFO:sim.faassim:setting up benchmark
INFO:__main__:autoscale-python-pi-cpu, latest, [ImageProperties(name='autoscale-python-pi-cpu', size=58000000, tag='latest', arch='arm32'), ImageProperties(name='autoscale-python-pi-cpu', size=58000000, tag='latest', arch='x86'), ImageProperties(name='autoscale-python-pi-cpu', size=58000000, tag='latest', arch='aarch64')]
INFO:sim.faassim:starting faas system
INFO:sim.faassim:starting be

## 4. 查看输出文件

样例运行完成后，会在下面目录生成 CSV 文件：

```text
examples/examples_autoscaling/outputs/
```

下面这个单元格用于列出输出文件。

In [4]:
output_dir = PROJECT_ROOT / "examples" / "examples_autoscaling" / "outputs"

print("输出目录：", output_dir, flush=True)

if not output_dir.exists():
    raise FileNotFoundError(f"输出目录不存在：{output_dir}")

csv_files = sorted(output_dir.glob("*.csv"))

print("CSV 文件数量：", len(csv_files), flush=True)

for path in csv_files:
    print(path.name, path.stat().st_size, "bytes", flush=True)

输出目录： C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\examples_autoscaling\outputs
CSV 文件数量： 8
autoscaling_replica_timeline.csv 79 bytes
autoscaling_summary.csv 122 bytes
flow.csv 135 bytes
function_deployment.csv 145 bytes
invocations.csv 28829 bytes
replica_deployment.csv 362 bytes
scale.csv 79 bytes
schedule.csv 362 bytes


## 5. 查看自动伸缩摘要

如果 `autoscaling_summary.csv` 已生成，可以直接读取查看。

In [5]:
import pandas as pd

summary_path = output_dir / "autoscaling_summary.csv"

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    display(summary_df)
else:
    print("未找到 autoscaling_summary.csv，请检查 main.py 是否运行完成。", flush=True)

,scale_events,invocation_events,schedule_events,replica_deployment_events,avg_exec_time
0,1,195,3,4,0.2


## 6. 说明

这个版本沿用你之前能正常运行的“命令行等价运行版”写法：

```python
process = subprocess.Popen(...)
for line in process.stdout:
    print(line, end="", flush=True)
```

它不会在 Notebook 中重新构造 `Simulation`，也不会用 `runpy` 直接执行脚本，而是尽量保持和 PowerShell 中运行 `.py` 的行为一致。